<a href="https://colab.research.google.com/github/sakinahdil/vlm-document-extraction/blob/main/vlm_document_extraction_without_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils
!pip install -q pdf2image
!apt-get install -y poppler-utils -q

In [ ]:
!git clone -q https://github.com/zzzDavid/ICDAR-2019-SROIE.git
import os, shutil
os.makedirs("/content/test_docs", exist_ok=True)
sroie = "/content/ICDAR-2019-SROIE/data/img"
for f in sorted(os.listdir(sroie))[:14]:
    shutil.copy(os.path.join(sroie, f), f"/content/test_docs/sroie_{f}")
print("Documents ready:", len(os.listdir("/content/test_docs")))

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", quantization_config=bnb, device_map="auto")
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
llava_model = LlavaForConditionalGeneration.from_pretrained("llava-hf/llava-1.5-7b-hf", quantization_config=bnb, device_map="auto")
llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
print("Both models loaded!")

In [ ]:
import os, json, re
import pandas as pd
from qwen_vl_utils import process_vision_info
from PIL import Image

def load_image(fp, max_side=1024):
    img = Image.open(fp).convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def extract_qwen(fp, q):
    img = load_image(fp)
    msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":q}]}]
    text = qwen_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ii,_ = process_vision_info(msgs)
    inp = qwen_processor(text=[text], images=ii, padding=True, return_tensors="pt").to("cuda")
    out = qwen_model.generate(**inp, max_new_tokens=256)
    raw = qwen_processor.decode(out[0][len(inp.input_ids[0]):], skip_special_tokens=True)
    try: return json.loads(re.sub(r"```json|```","",raw).strip())
    except: return {"_raw": raw}

def extract_llava(fp, q):
    img = load_image(fp)
    inp = llava_processor(images=img, text=f"USER: <image>\n{q}\nASSISTANT:", return_tensors="pt").to("cuda")
    out = llava_model.generate(**inp, max_new_tokens=256)
    ans = llava_processor.decode(out[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
    try: return json.loads(re.sub(r"```json|```","",ans).strip())
    except: return {"_raw": ans}

folder = "/content/test_docs"
files = sorted([f for f in os.listdir(folder) if f.lower().endswith((".png",".jpg",".jpeg"))])
prompt = "Extract the following fields from this document and return them as JSON: vendor name, date, invoice or receipt number, total amount (as a number), and currency. If a field is missing, use null."

rows = []
for fn in files:
    p = os.path.join(folder, fn)
    print(f"Processing: {fn}")
    try: qo = extract_qwen(p, prompt)
    except Exception as e: qo = {"_error": str(e)[:60]}
    try: lo = extract_llava(p, prompt)
    except Exception as e: lo = {"_error": str(e)[:60]}
    rows.append({"document":fn,
                 "qwen_valid":"_raw" not in qo and "_error" not in qo,
                 "llava_valid":"_raw" not in lo and "_error" not in lo,
                 "qwen_output":str(qo), "llava_output":str(lo)})
df = pd.DataFrame(rows)
df.to_excel("/content/model_comparison.xlsx", index=False)
print("\nSaved to /content/model_comparison.xlsx")

from google.colab import files
files.download("/content/model_comparison.xlsx")

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

folder = "/content/test_docs"
files = sorted([f for f in os.listdir(folder) if f.lower().endswith((".png",".jpg",".jpeg"))])

for fn in files:
    img = Image.open(os.path.join(folder, fn))
    plt.figure(figsize=(6, 8))
    plt.imshow(img)
    plt.title(fn, fontsize=10)
    plt.axis("off")
    plt.show()

In [ ]:
print("qwen_model" in dir())
print("extract_qwen" in dir())
print("process_and_prepare" in dir())

In [ ]:
# ---- Install (only what the app needs) ----
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils gradio pdf2image
!apt-get install -y poppler-utils -q

# ---- Load Qwen model (app uses Qwen only) ----
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", quantization_config=bnb, device_map="auto")
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
print("Model loaded!")

# ---- Extraction + app functions ----
import json, re, os, tempfile
import gradio as gr
from qwen_vl_utils import process_vision_info
from PIL import Image
from pdf2image import convert_from_path

def load_image(fp, max_side=1024):
    if fp.lower().endswith(".pdf"):
        img = convert_from_path(fp, dpi=200)[0].convert("RGB")
    else:
        img = Image.open(fp).convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def extract_qwen(fp, q):
    img = load_image(fp)
    msgs = [{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":q}]}]
    text = qwen_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ii,_ = process_vision_info(msgs)
    inp = qwen_processor(text=[text], images=ii, padding=True, return_tensors="pt").to("cuda")
    out = qwen_model.generate(**inp, max_new_tokens=256)
    raw = qwen_processor.decode(out[0][len(inp.input_ids[0]):], skip_special_tokens=True)
    try: return json.loads(re.sub(r"```json|```","",raw).strip())
    except: return {"_raw": raw}

def process_and_prepare(file_path):
    if file_path is None:
        return None, [["No file", "Please upload a document"]], None
    prompt = "Extract the following fields from this document and return them as JSON: vendor name, date, invoice or receipt number, total amount (as a number), and currency. If a field is missing, use null."
    result = extract_qwen(file_path, prompt)
    if "_raw" in result or "_error" in result:
        table = [["Raw output", str(result)]]; json_data = result
    else:
        labels = {"vendor_name":"Vendor Name","date":"Date","invoice_number":"Invoice / Receipt No.","invoice_or_receipt_number":"Invoice / Receipt No.","total_amount":"Total Amount","currency":"Currency"}
        table = [[labels.get(k,k), str(v)] for k,v in result.items()]; json_data = result
    out_path = os.path.join(tempfile.gettempdir(), "extracted_data.json")
    with open(out_path,"w") as f: json.dump(json_data, f, indent=2)
    return file_path, table, out_path

# ---- Launch the app ----
with gr.Blocks(title="Document Information Extractor") as demo:
    gr.Markdown("# 📄 Document Information Extractor")
    gr.Markdown("Upload a receipt, invoice, or bill (image or PDF). The system uses Qwen-VL to extract the key information automatically.")
    with gr.Row():
        file_input = gr.File(label="Upload Document", type="filepath")
    extract_button = gr.Button("Extract Information", variant="primary")
    with gr.Row():
        image_preview = gr.Image(label="Uploaded Document", type="filepath")
        output_table = gr.Dataframe(headers=["Field","Value"], label="Extracted Information", wrap=True)
    download_file = gr.File(label="Download Extracted Data (JSON)")
    extract_button.click(fn=process_and_prepare, inputs=file_input, outputs=[image_preview, output_table, download_file])

demo.launch(share=True)